# Lab 10 — Tokenization & Answer Span Alignment

Notebook này kiểm tra từng bước biến một mẫu SQuAD `(question, context, answer_text, answer_start)` thành nhãn
`(start_positions, end_positions)` ở mức token:

1. Cấu trúc input `[CLS] question [SEP] context [SEP]`: `input_ids`, `attention_mask`, `token_type_ids`, `sequence_ids`
2. Character index ↔ word index ↔ token index (Calculation Exercise 3)
3. `offset_mapping` và thuật toán căn chỉnh character span → token span
4. Các trường hợp biên: answer bắt đầu/kết thúc giữa token, answer bị truncation, answer nằm ngoài window
5. Sliding window: `stride`, `return_overflowing_tokens`, `overflow_to_sample_mapping`
6. Thống kê độ dài context/answer của SQuAD và kiểm tra hàm EM/F1

In [1]:
import collections
import numpy as np
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer
from qa_model import char_to_token_span, prepare_train_features, squad_em, squad_f1, normalize_answer

pd.set_option('display.width', 200, 'display.max_columns', 30)
tokenizer = AutoTokenizer.from_pretrained('bert-base-cased')
squad = load_dataset('rajpurkar/squad')
print(squad)

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 87599
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 10570
    })
})


## 1. Cấu trúc input của QA

In [2]:
question = 'What is the capital of France?'
context = 'The capital of France is Paris.'
enc = tokenizer(question, context, return_offsets_mapping=True)

df = pd.DataFrame({
    'token': tokenizer.convert_ids_to_tokens(enc['input_ids']),
    'input_id': enc['input_ids'],
    'attention_mask': enc['attention_mask'],
    'token_type_id': enc['token_type_ids'],
    'sequence_id': enc.sequence_ids(),
    'offset': enc['offset_mapping'],
})
df.T

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16
token,[CLS],What,is,the,capital,of,France,?,[SEP],The,capital,of,France,is,Paris,.,[SEP]
input_id,101,1327,1110,1103,2364,1104,1699,136,102,1109,2364,1104,1699,1110,2123,119,102
attention_mask,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
token_type_id,0,0,0,0,0,0,0,0,0,1,1,1,1,1,1,1,1
sequence_id,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,1.0,1.0,1.0,1.0,1.0,1.0,1.0,NaN
offset,"(0, 0)","(0, 4)","(5, 7)","(8, 11)","(12, 19)","(20, 22)","(23, 29)","(29, 30)","(0, 0)","(0, 3)","(4, 11)","(12, 14)","(15, 21)","(22, 24)","(25, 30)","(30, 31)","(0, 0)"


**Nhận xét:**
- `input_ids`: id trong vocab của `bert-base-cased` (101 = `[CLS]`, 102 = `[SEP]`).
- `attention_mask`: 1 cho token thật, 0 cho padding (ở đây chưa padding nên toàn 1).
- `token_type_ids`: 0 cho segment A (`[CLS]` + question + `[SEP]` đầu), 1 cho segment B (context + `[SEP]` cuối).
- `sequence_ids()`: `None` cho special tokens, 0 = question, 1 = context. **Đây** mới là thứ ta dùng để biết token nào thuộc context — không dựa vào `token_type_ids`.
- `offset_mapping`: vị trí ký tự `(start, end)` **trong chuỗi gốc tương ứng** (question cho phần 0, context cho phần 1); special tokens có `(0, 0)`.

Không phải Transformer nào cũng có `token_type_ids`: RoBERTa/DistilBERT không dùng segment embedding.

In [3]:
for name in ['bert-base-cased', 'roberta-base', 'distilbert-base-cased']:
    t = AutoTokenizer.from_pretrained(name)
    e = t(question, context)
    print(f"{name:24s} keys={list(e.keys())}")
    print(' ' * 24, t.convert_ids_to_tokens(e['input_ids']))

bert-base-cased          keys=['input_ids', 'token_type_ids', 'attention_mask']
                         ['[CLS]', 'What', 'is', 'the', 'capital', 'of', 'France', '?', '[SEP]', 'The', 'capital', 'of', 'France', 'is', 'Paris', '.', '[SEP]']


roberta-base             keys=['input_ids', 'attention_mask']
                         ['<s>', 'What', 'Ġis', 'Ġthe', 'Ġcapital', 'Ġof', 'ĠFrance', '?', '</s>', '</s>', 'The', 'Ġcapital', 'Ġof', 'ĠFrance', 'Ġis', 'ĠParis', '.', '</s>']


distilbert-base-cased    keys=['input_ids', 'token_type_ids', 'attention_mask']
                         ['[CLS]', 'What', 'is', 'the', 'capital', 'of', 'France', '?', '[SEP]', 'The', 'capital', 'of', 'France', 'is', 'Paris', '.', '[SEP]']


RoBERTa dùng `<s> question </s></s> context </s>` và **không** trả về `token_type_ids`. Tokenizer của DistilBERT vẫn sinh `token_type_ids` (do dùng chung WordPiece của BERT) nhưng `DistilBertModel` **không có segment embedding** và không nhận tham số này — tức là không thể suy ra việc model có dùng `token_type_ids` chỉ từ output của tokenizer.
Vì vậy code trong `qa_model.py` chỉ đưa `token_type_ids` vào model nếu tokenizer có sinh ra (`MODEL_INPUTS` được lọc theo cột có thật),
và xác định phần context bằng `sequence_ids()` (hoạt động cho mọi tokenizer fast).

## 2. Calculation Exercise 3 — character index vs word index vs token index

In [4]:
answer = 'Paris'
answer_start = context.index(answer)
answer_end = answer_start + len(answer)
print('characters :', list(enumerate(context)))
print(f'answer_start (first char) = {answer_start}, last char = {answer_end - 1}, exclusive end = {answer_end}')
print('context[answer_start:answer_end] =', repr(context[answer_start:answer_end]))

words = context.split()
print('word index of Paris (whitespace split):', [i for i, w in enumerate(words) if 'Paris' in w], '->', words)

s, e, status = char_to_token_span(enc['offset_mapping'], enc.sequence_ids(), answer_start, answer_end)
toks = tokenizer.convert_ids_to_tokens(enc['input_ids'])
print(f'start_token={s} ({toks[s]}), end_token={e} ({toks[e]}), status={status}')
print('decoded:', tokenizer.decode(enc['input_ids'][s:e + 1]))

characters : [(0, 'T'), (1, 'h'), (2, 'e'), (3, ' '), (4, 'c'), (5, 'a'), (6, 'p'), (7, 'i'), (8, 't'), (9, 'a'), (10, 'l'), (11, ' '), (12, 'o'), (13, 'f'), (14, ' '), (15, 'F'), (16, 'r'), (17, 'a'), (18, 'n'), (19, 'c'), (20, 'e'), (21, ' '), (22, 'i'), (23, 's'), (24, ' '), (25, 'P'), (26, 'a'), (27, 'r'), (28, 'i'), (29, 's'), (30, '.')]
answer_start (first char) = 25, last char = 29, exclusive end = 30
context[answer_start:answer_end] = 'Paris'
word index of Paris (whitespace split): [5] -> ['The', 'capital', 'of', 'France', 'is', 'Paris.']
start_token=14 (Paris), end_token=14 (Paris), status=ok
decoded: Paris


In [5]:
# Đổi answer thành "capital of France" (Calculation Exercise 1b)
a = 'capital of France'
a_s = context.index(a); a_e = a_s + len(a)
s, e, _ = char_to_token_span(enc['offset_mapping'], enc.sequence_ids(), a_s, a_e)
print(f'char span=[{a_s},{a_e}) -> token span=({s},{e}) = {toks[s:e+1]}')
print('Trong riêng context (bỏ [CLS] + 7 token question + [SEP] = 9 token):', (s - 9, e - 9))

char span=[4,21) -> token span=(10,12) = ['capital', 'of', 'France']
Trong riêng context (bỏ [CLS] + 7 token question + [SEP] = 9 token): (1, 3)


Ba loại chỉ số của cùng đáp án "Paris": **character** 25→30, **word** 5 (`'Paris.'` dính dấu chấm), **token** 14.
Offset mapping là cầu nối duy nhất đáng tin cậy giữa ký tự và token.

## 3. Thuật toán căn chỉnh trên dữ liệu SQuAD thật

Hàm `char_to_token_span` (trong `qa_model.py`) cài đặt đúng pseudo-code của đề:

```python
for token_idx, (start, end) in enumerate(offsets):       # chỉ duyệt token thuộc context
    if start <= answer_start < end: start_token = token_idx
    if start < answer_end <= end:  end_token = token_idx
```
và trả về `(0, 0)` (trỏ vào `[CLS]`) nếu answer không nằm trọn trong window.

In [6]:
ex = squad['train'][0]
a_text, a_start = ex['answers']['text'][0], ex['answers']['answer_start'][0]
a_end = a_start + len(a_text)
enc = tokenizer(ex['question'], ex['context'], return_offsets_mapping=True, truncation='only_second', max_length=384)
s, e, status = char_to_token_span(enc['offset_mapping'], enc.sequence_ids(), a_start, a_end)
toks = tokenizer.convert_ids_to_tokens(enc['input_ids'])
print('Q:', ex['question']); print('Gold:', a_text, '| chars', (a_start, a_end))
print('token span:', (s, e), toks[s:e + 1])
print('offsets of span:', enc['offset_mapping'][s:e + 1])
recovered = ex['context'][enc['offset_mapping'][s][0]:enc['offset_mapping'][e][1]]
print('recovered from offsets:', repr(recovered), '| match =', recovered == a_text)

Q: To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?
Gold: Saint Bernadette Soubirous | chars (515, 541)
token span: (136, 142) ['Saint', 'Bern', '##ade', '##tte', 'So', '##ubi', '##rous']
offsets of span: [(515, 520), (521, 525), (525, 528), (528, 531), (532, 534), (534, 537), (537, 541)]
recovered from offsets: 'Saint Bernadette Soubirous' | match = True


### Kiểm tra toàn bộ tập train: answer khôi phục từ token span có khớp gold không?

In [7]:
def audit(split, max_length=384, stride=128, n=None):
    data = squad[split] if n is None else squad[split].select(range(n))
    stats = collections.Counter(); mid_examples = []
    for ex in data:
        a_text, a_start = ex['answers']['text'][0], ex['answers']['answer_start'][0]
        a_end = a_start + len(a_text)
        enc = tokenizer(ex['question'].lstrip(), ex['context'], truncation='only_second', max_length=max_length,
                        stride=stride, return_overflowing_tokens=True, return_offsets_mapping=True)
        found = False
        for w in range(len(enc['input_ids'])):
            offs, seq = enc['offset_mapping'][w], enc.sequence_ids(w)
            s, e, status = char_to_token_span(offs, seq, a_start, a_end)
            stats['windows'] += 1
            stats['window_' + status] += 1
            if status == 'ok' and not found:
                found = True
                rec = ex['context'][offs[s][0]:offs[e][1]]
                stats['exact_recover'] += rec == a_text
                stats['normalized_recover'] += normalize_answer(rec) == normalize_answer(a_text)
                mid_s, mid_e = offs[s][0] != a_start, offs[e][1] != a_end
                stats['start_mid_token'] += mid_s; stats['end_mid_token'] += mid_e
                if (mid_s or mid_e) and len(mid_examples) < 8:
                    mid_examples.append({'gold': a_text, 'recovered': rec,
                                         'tokens': tokenizer.convert_ids_to_tokens(enc['input_ids'][w][s:e + 1]),
                                         'start_mid': mid_s, 'end_mid': mid_e})
        stats['examples'] += 1
        stats['answer_found_in_some_window'] += found
    return stats, mid_examples

stats, mid_examples = audit('train')
total = stats['examples']
print({k: v for k, v in stats.items()})
for k in ['answer_found_in_some_window', 'exact_recover', 'start_mid_token', 'end_mid_token']:
    print(f"{k:30s}: {stats[k]:6d} / {total} = {100 * stats[k] / total:.3f}%")
pd.DataFrame(mid_examples)

{'windows': 88729, 'window_ok': 87823, 'exact_recover': 87379, 'normalized_recover': 87379, 'start_mid_token': 82, 'end_mid_token': 150, 'examples': 87599, 'answer_found_in_some_window': 87599, 'window_outside': 906}
answer_found_in_some_window   :  87599 / 87599 = 100.000%
exact_recover                 :  87379 / 87599 = 99.749%
start_mid_token               :     82 / 87599 = 0.094%
end_mid_token                 :    150 / 87599 = 0.171%


,gold,recovered,tokens,start_mid,end_mid
0,split with Luckett and Rober,split with Luckett and Roberson,"[split, with, Luck, ##ett, and, Rob, ##erson]",False,True
1,six,sixth,[sixth],False,True
2,evolution,evolutionary,[evolutionary],False,True
3,7,1817,[1817],True,False
4,7,1817,[1817],True,False
5,7,1817,[1817],True,False
6,opera,operatic,[operatic],False,True
7,Tibet,Tibetan,[Tibetan],False,True


**Trường hợp answer bắt đầu/kết thúc giữa token.** Một số ít gold answer cắt ngang một token (ví dụ gold `"1"` trong token `"1st"`,
hoặc gold không lấy hậu tố `##s`). Theo quy tắc `start <= answer_start < end`, token chứa ký tự biên được chọn, nên khi giải mã lại
bằng offset ta thu được **cả token** → span dài hơn gold một chút. Đây là giới hạn cứng của mọi model span-prediction trên subword:
những mẫu này không bao giờ đạt EM bằng cách giải mã thông thường (nhưng vẫn được F1 một phần). Tỷ lệ rất nhỏ (xem số liệu trên).

## 4. Truncation & sliding window

In [8]:
long_ex = max(squad['validation'].select(range(2000)), key=lambda x: len(x['context']))
print('context chars:', len(long_ex['context']), '| context tokens:',
      len(tokenizer(long_ex['context'])['input_ids']))
print('Q:', long_ex['question']); print('Gold:', long_ex['answers'])

inputs = tokenizer(long_ex['question'], long_ex['context'], truncation='only_second', max_length=384, stride=128,
                   return_overflowing_tokens=True, return_offsets_mapping=True)
print('keys:', list(inputs.keys()))
print('number of windows:', len(inputs['input_ids']))
print('overflow_to_sample_mapping:', inputs['overflow_to_sample_mapping'])
a_start = long_ex['answers']['answer_start'][0]; a_end = a_start + len(long_ex['answers']['text'][0])
rows = []
for w in range(len(inputs['input_ids'])):
    offs, seq = inputs['offset_mapping'][w], inputs.sequence_ids(w)
    ctx = [o for o, s_ in zip(offs, seq) if s_ == 1]
    s, e, status = char_to_token_span(offs, seq, a_start, a_end)
    rows.append({'window': w, 'n_tokens': len(inputs['input_ids'][w]), 'ctx_char_range': (ctx[0][0], ctx[-1][1]),
                 'n_ctx_tokens': len(ctx), 'label': (s, e), 'status': status,
                 'decoded': tokenizer.decode(inputs['input_ids'][w][s:e + 1]) if status == 'ok' else '[CLS]'})
pd.DataFrame(rows)

context chars: 2060 | context tokens: 420
Q: Who is the head coach of the Broncos?
Gold: {'text': ['Gary Kubiak', 'Gary Kubiak', 'Kubiak'], 'answer_start': [291, 291, 332]}
keys: ['input_ids', 'token_type_ids', 'attention_mask', 'offset_mapping', 'overflow_to_sample_mapping']
number of windows: 2
overflow_to_sample_mapping: [0, 0]


,window,n_tokens,ctx_char_range,n_ctx_tokens,label,status,decoded
0,0,384,"(0, 1810)",372,"(64, 67)",ok,Gary Kubiak
1,1,186,"(1177, 2060)",174,"(0, 0)",outside,[CLS]


In [9]:
# Overlap giữa 2 window liên tiếp = stride token (tính trên phần context)
w0 = [o for o, s_ in zip(inputs['offset_mapping'][0], inputs.sequence_ids(0)) if s_ == 1]
w1 = [o for o, s_ in zip(inputs['offset_mapping'][1], inputs.sequence_ids(1)) if s_ == 1]
overlap = [o for o in w1 if o in set(w0)]
print('context tokens in window 0:', len(w0), '| window 1:', len(w1), '| shared tokens:', len(overlap))
print('window 1 starts at char', w1[0][0], '= window 0 char', w0[-len(overlap)][0])

context tokens in window 0: 372 | window 1: 174 | shared tokens: 128
window 1 starts at char 1177 = window 0 char 1177


**Giải thích các trường trả về:**
- `input_ids` / `attention_mask` / `token_type_ids`: mỗi window là một feature riêng, question được lặp lại ở mọi window.
- `offset_mapping`: offset **luôn tính theo context gốc**, nên ta biết window nào phủ đoạn ký tự nào.
- `overflow_to_sample_mapping`: feature *i* thuộc example nào — cần để gom các window về lại một câu hỏi khi đánh giá.
- `stride=128`: số token context **chồng lấn** giữa hai window liên tiếp, đảm bảo một answer nằm gần biên window vẫn xuất hiện
  trọn vẹn trong ít nhất một window.
- Window không chứa trọn answer được gán nhãn `(0, 0)` = `[CLS]` ("không có đáp án trong window này").

### Thu nhỏ `max_length` để thấy rõ ảnh hưởng của truncation

In [10]:
sub = squad['validation'].select(range(2000))
rows = []
for L in [128, 256, 384]:
    feats = sub.map(prepare_train_features, batched=True, remove_columns=sub.column_names,
                    fn_kwargs=dict(tokenizer=tokenizer, max_length=L, stride=L // 3))
    inw = np.array(feats['answer_in_window'])
    rows.append({'max_length': L, 'stride': L // 3, 'features': len(feats),
                 'windows/example': round(len(feats) / len(sub), 3),
                 '% windows containing answer': round(100 * inw.mean(), 2)})
pd.DataFrame(rows)

,max_length,stride,features,windows/example,% windows containing answer
0,128,42,3796,1.898,60.51
1,256,85,2180,1.090,93.07
2,384,128,2028,1.014,98.82


Với `max_length=128` mỗi câu hỏi bị chia thành nhiều window hơn, phần lớn window **không** chứa đáp án (nhãn `[CLS]`).
Nếu chỉ `truncation` mà không dùng `return_overflowing_tokens`, các answer nằm sau token thứ ~110 của context sẽ **mất hẳn**.

## 5. Thống kê độ dài trong SQuAD

In [11]:
val = squad['validation']
ctx_len = np.array([len(x) for x in tokenizer(list(val['context']))['input_ids']])
q_len = np.array([len(x) for x in tokenizer(list(val['question']))['input_ids']])
ans_words = np.array([len(a['text'][0].split()) for a in val['answers']])
print(f'context tokens: mean={ctx_len.mean():.1f} median={np.median(ctx_len):.0f} max={ctx_len.max()}')
print(f'question tokens: mean={q_len.mean():.1f} max={q_len.max()}')
for L in [128, 256, 384]:
    print(f'  % questions whose question+context > {L}: {100 * ((ctx_len + q_len) > L).mean():.1f}%')
print('answer length (words) buckets:',
      collections.Counter(np.select([ans_words == 1, ans_words <= 3, ans_words <= 6], ['1', '2-3', '4-6'], '7+')))

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (627 > 512). Running this sequence through the model will result in indexing errors


context tokens: mean=167.8 median=153 max=805
question tokens: mean=14.9 max=44
  % questions whose question+context > 128: 81.6%
  % questions whose question+context > 256: 12.5%
  % questions whose question+context > 384: 2.1%
answer length (words) buckets: Counter({np.str_('2-3'): 4575, np.str_('1'): 3429, np.str_('4-6'): 1648, np.str_('7+'): 918})


## 6. Kiểm tra hàm metric với các ví dụ tính tay (calculations.md)

In [12]:
cases = [('France', 'Paris France'), ('the city of Paris', 'Paris'), ('New York', 'New York City'),
         ('Paris.', 'the Paris'), ('Paris', 'Paris')]
pd.DataFrame([{'prediction': p, 'gold': g, 'norm_pred': normalize_answer(p), 'norm_gold': normalize_answer(g),
               'EM': squad_em(p, g), 'F1': round(squad_f1(p, g), 3)} for p, g in cases])

,prediction,gold,norm_pred,norm_gold,EM,F1
0,France,Paris France,france,paris france,0.0,0.667
1,the city of Paris,Paris,city of paris,paris,0.0,0.500
2,New York,New York City,new york,new york city,0.0,0.800
3,Paris.,the Paris,paris,paris,1.0,1.000
4,Paris,Paris,paris,paris,1.0,1.000


Kết quả khớp với phần tự tính trong `calculations.md` (0.667, 0.500, 0.800, EM=1 sau chuẩn hóa).